# Micro-Hay auxiliary phase fine-tuning 05

This iteration keeps the input-only GRU and its 61-state decoder unchanged at inference. A training-only linear auxiliary head applies deep supervision to suprathreshold occupancy and rapid voltage phase. The physical objective remains symmetric and adds explicit soma distillation outside teacher spike windows. Checkpoint admission uses physical rollout constraints only.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/kaggle/working/LearningSingleCompartiment')
if not (ROOT / 'pyproject.toml').exists():
    if ROOT.exists() and any(ROOT.iterdir()): raise RuntimeError(f'{ROOT} exists but is not the project')
    subprocess.check_call(['git', 'clone', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', 'main'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

## Inputs
Mount `hay_micro_4c_event_enriched_v2.h5` and `gru_mse.pt` through **Add Input**. The notebook searches nested Kaggle input directories automatically and never regenerates the dataset.

In [ ]:
def find_kaggle_input(working_path, pattern, label):
    working_path = Path(working_path)
    candidates = [working_path] if working_path.exists() else []
    if Path('/kaggle/input').exists(): candidates += sorted(Path('/kaggle/input').rglob(pattern))
    if not candidates: raise FileNotFoundError(f'{label} non trovato. Montalo con Add Input: {pattern}')
    selected = candidates[0]
    print(f'{label}:', selected, f'({selected.stat().st_size/2**20:.1f} MiB)')
    return selected

DATASET = find_kaggle_input('/kaggle/working/hay_micro_4c_event_enriched_v2.h5', 'hay_micro_4c_event_enriched_v2*.h5', 'Dataset')
BASELINE = find_kaggle_input('/kaggle/working/gru_mse.pt', 'gru_mse.pt', 'Checkpoint GRU-MSE')
os.environ['HAY_FINETUNE_DATASET'] = str(DATASET)
os.environ['HAY_FINETUNE_BASELINE'] = str(BASELINE)
os.environ['HAY_FINETUNE_OUTPUT'] = '/kaggle/working/hay_micro_phase_auxiliary_finetune_05'
os.environ['HAY_FINETUNE_OBJECTIVE'] = 'phase_auxiliary_v5'
os.environ['HAY_FINETUNE_EPOCHS'] = '30'
os.environ['HAY_FINETUNE_PATIENCE'] = '12'
os.environ['HAY_FINETUNE_LEARNING_RATE'] = '1e-5'
os.environ['HAY_FINETUNE_CURRICULUM_EPOCHS'] = '8'
os.environ['HAY_FINETUNE_WINDOWS_PER_EPOCH'] = '48'
os.environ['HAY_FINETUNE_FORCE_RESTART'] = '0'
%run /kaggle/working/LearningSingleCompartiment/notebooks/micro_spike_finetune_03.py

## Audit selected and rejected models
The comparison includes GRU-MSE, the validation-selected checkpoint and the last rejected candidate when they differ. `selected_epoch = 0` means that every fine-tuned epoch violated a constraint or failed to improve the physical objective.

In [ ]:
import pandas as pd
from IPython.display import Image, display
RESULTS = Path('/kaggle/working/hay_micro_phase_auxiliary_finetune_05')
display(pd.read_csv(RESULTS / 'comparison.csv').T)
history = pd.read_csv(RESULTS / 'finetune_history.csv')
columns = [c for c in ['epoch','event_scale','checkpoint_admissible','validation_global','validation_subthreshold_rmse_mV','validation_predicted_above_steps','validation_predicted_crossings','validation_waveform','validation_derivative','validation_soma_reference','validation_auxiliary_bce','validation_auxiliary_derivative'] if c in history]
display(history[columns].tail(30))
display(Image(str(RESULTS / 'soma_comparison.png')))

## Export complete evidence
The archive includes best and last checkpoints, training history, physical predictions and plots.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, FileLink, display
zip_path = Path(make_archive('/kaggle/working/hay_micro_phase_auxiliary_finetune_05_complete', 'zip', root_dir=RESULTS.parent, base_dir=RESULTS.name))
print('Archive:', zip_path, f'({zip_path.stat().st_size/2**20:.1f} MiB)')
display(FileLink(str(zip_path)))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{encoded}'),x=new Uint8Array(b.length);for(let i=0;i<b.length;i++)x[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([x],{{type:'application/zip'}})),a=document.createElement('a');a.href=u;a.download='{zip_path.name}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(u),60000);"""))
print('Download avviato:', zip_path.name)